# Initial setup 
As with all of PyEBSDIndex, it is recommended to set up a conda environment for PyEBSDIndex.  If you have conda running on your machine, it _should_ be as easy as: 

```    
conda create --name nlstem python=3.11 pyebsdindex jupyter 
conda activate nlstem
```

You can use python 3.9 <--> 3.12 (and maybe 3.13?).  This should install all the necessary dependacies for pyebsdindex (with nlpar/nlstem).  

NOTE: throughout this notebook, ```{Text in braces}``` will indicate information the user should fill in, removing the braces afterward.  

In [ ]:
import sys, os

# This maybe necessary if using a development version, such as one downloaded 
# from https://github.com/drowenhorst-nrl/PyEBSDIndex/tree/develop   
# sys.path.insert(0,r'/{Path to PyEBSDIndex folder}/PyEBSDIndex/')
import numpy as np
import scipy
import matplotlib.pyplot as plt
import h5py
from pyebsdindex import ebsd_pattern
from pyebsdindex import nlpar_cpu as nlpar # recommended for NLSTEM 
#from pyebsdindex.opencl import nlpar_cl as nlpar ### experiemental for NLSTEM, depending on GPU/Hardware might be much faster. 
#from pyebsdindex import nlpar ### this will autodetect if a GPU is present, and default to using it.  


### Initial data read for making a mask
This will laod the DM5 file as a pattern file, reading off some of the relevant header info.  Pattern reading is done in a separate command.  

In [ ]:
file = '/{Path to DM5 file}/{DM5 filename}.dm5'
fid = ebsd_pattern.get_pattern_file_obj(file)

### Read in some patterns, and make a mask that excludes the transmitted peak.

In [ ]:
imsize = (fid.nRows, fid.nCols) # nRows, nCols are the number patterns in the scan grid.  

sample_pats, loc = fid.read_data(patStartCount=[ [imsize[0]//2-5, imsize[1]//2-5], [10,10]], returnArrayOnly=True) 
# read some patterns from the data. Format is [[colstart, rowstart], [numcolpatterns, numrowpatterns]] 
# Here we read in 25 patterns --> this maybe should be adjusted to make sure multiple grains are sampled.  Reading in all patterns would be overkill.  

# Get a mean pattern.
meanpat = sample_pats.mean(axis=0) # pull a mean pattern 

maxpat = meanpat> (meanpat.max()*0.75) # filter out the highest values.  
maxpatL, npeaks = scipy.ndimage.label(maxpat.astype(int)) # label them (might be more than one peak identified)
#plt.imshow(maxpatL)
cm = scipy.ndimage.center_of_mass(maxpat, maxpatL, index = np.arange(npeaks)+1) # and measure their peak center of mass. 

peakloc = np.array(cm[:][0]) # initialize some values
peakmx = meanpat[maxpatL == 1].mean()

# catch the case where multiple max peaks are found --- keep the one that has the higest max and assume that in a set of average patterns, that will be the transmitted beam.  
for ii in range(npeaks):
  peakloc1 = np.array(cm[:][ii]) 
  peakmx1 = meanpat[maxpatL == ii+1].mean()
  if peakmx1 > peakmx:
    peakloc = peakloc1
    peakmx = peakmx1

# make a circular mask that is centered on the tranmitted peak. 
masky, maskx = np.unravel_index(np.arange(fid.patternH * fid.patternW, dtype=int), (fid.patternH, fid.patternW)) 
mask = np.sqrt( (maskx-peakloc[1])**2 + (masky-peakloc[0])**2).reshape(fid.patternH, fid.patternW) > 25 # the radius is probably not perfect, but should work well enough.  


fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))
ax[0].imshow(meanpat)
ax[1].imshow(mask)


### Set up NLPAR object

In [ ]:
nlparobj = nlpar.NLPAR(file, mask=mask) # I assign the pattern mask when I make the nlpar object.  
##However, one can assign the mask later by doing 

#nlparonj.mask = mask 
## where mask must be a 2D numpy array of ones and zeros that is the same size as a single pattern.  

# the stem_scale keyword here is new, and important.  It will scale all the patterns to pat = np.sqrt(pat-pat.min()) for the calcualtions.
lam = nlparobj.opt_lambda(saturation_protect=True, stem_scale=True) 
#As inthe NLPAR program, this will attempt to precalculate an appropriate lambda value for the data,
# assuming that most the pattern neighbors have a high probability of being the same.
#The lambda values will be used later in the NLPAR calculation.  

#Three lambda choices are given in lam=[less blending, average amount, more], and I am going to choose the average amount.  
print(nlparobj.sigma.min(), nlparobj.sigma.mean(), nlparobj.sigma.max())
plt.imshow(nlparobj.sigma) # sigma plot is inidicative of the amount of noise in each pattern, and approximates a pattern quality map in EBSD.  
# The sigma values are saved, and used later in the nlpar calculation.

The return of the ```opt_lambda``` routine will be the three optimized lambda values. It is recommended to always use ```saturation_protect=True```.  One can try both ```stem_scale=True/False```, but the lambda and sigma values are specific to how this is set.

In [ ]:
# again stem_scale keyword here is new, and important. We will note, the patterns are scaled back before being written out with patout = (nlpat)**2+pat0.min().
nlstemfile = nlparobj.calcnlpar(stem_scale=True, saturation_protect = True, lam = lam[1], searchradius = 4) # this actually runs NLPAR calculations.
# the nlstemfile will hold the filepath that is the output data.  At this point, the program makes a copy of the original file (adding some info to the file name), and overwrites the original pattern data.

One can take the new nlstem file and index it using your method of choice.  Below is just some code to inspect the results.

In [ ]:
fidout = ebsd_pattern.get_pattern_file_obj(nlstemfile) 
out_pats, loc = fidout.read_data(patStartCount=[ [imsize[0]//2-5, imsize[1]//2-5], [10,10]], returnArrayOnly=True) # read some patterns from the data. Format is [[colstart, rowstart], [numcolpatterns, numrowpatterns]] 

In [ ]:
import PIL

In [ ]:

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 6))
ii = 50 # chose which pattern out of the 25 to look at.  
contrast=0 # [values of 1, 10, 100, 1000 to increase contrast ratio for visualization]
im0 = (sample_pats[ii,:,:])
im1 = out_pats[ii,:,:]
axes[0,2].imshow(np.abs(im0-im1))

mx = max(im0.max(), im1.max())
mn = min(im0.min(), im1.min())

#PIL.Image.fromarray((im0-mn)/(mx-mn), mode='F').save('./im0raw.tif')
#PIL.Image.fromarray((im1-mn)/(mx-mn), mode='F').save('./im1raw.tif')

x0, y0 = 229,503 # These are in _pixel_ coordinates -- adjust for your own line profile analysis. 
x1, y1 = 457,73
num = max(abs(x0-x1), abs(y0-y1))
x, y = np.linspace(x0, x1, num), np.linspace(y0, y1, num)

# Extract the values along the line, using cubic interpolation
zi0 = scipy.ndimage.map_coordinates(np.transpose(im0), np.vstack((x,y))) # THIS SEEMS TO WORK CORRECTLY
zi0 = (zi0 - mn)/(mx-mn)
zi1 = scipy.ndimage.map_coordinates(np.transpose(im1), np.vstack((x,y))) # THIS SEEMS TO WORK CORRECTLY
zi1 = (zi1 - mn)/(mx-mn)
dat0 = np.array([x,y,zi0])
dat1 = np.array([x,y,zi1])

#np.savetxt('profile0.csv', np.transpose(dat0), delimiter=",")
#np.savetxt('profile1.csv', np.transpose(dat1), delimiter=",")
axes[1,0].plot(zi0)
axes[1,1].plot(zi1)


#print(im0.min(), im0.max())
im0 = np.log(im0-mn+1+contrast) # do a bit of contrast enhancement.  
#print(im1.min(), im1.max())
im1 = np.log(im1-mn+1+contrast)
axes[0,0].imshow(im0)
axes[0,1].imshow(im1)
#plt.imsave('pat0.png', im0)
#plt.imsave('pat0nlstem.png', im1)

